In [3]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split

class FeatureEngineer:

    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()
        self._scalers = {}
        self._encoders = {}



    def minmax_scale(self, columns: list) -> "FeatureEngineer":

        for col in columns:
            scaler = MinMaxScaler()
            self.df[col] = scaler.fit_transform(self.df[[col]])
            self._scalers[col] = scaler
        return self



    def standard_scale(self, columns: list) -> "FeatureEngineer":
        
        for col in columns:
            scaler = StandardScaler()
            self.df[col] = scaler.fit_transform(self.df[[col]])
            self._scalers[col] = scaler
        return self



    def log_transform(self, columns: list) -> "FeatureEngineer":
        
        for col in columns:
            self.df[col] = np.log1p(self.df[col])
        return self

    def onehot_encode(self, columns: list, drop_first: bool = True) -> "FeatureEngineer":
        self.df = pd.get_dummies(self.df, columns=columns, drop_first=drop_first)
        return self



    def equal_width_bin(self, columns: list, bins: int) -> "FeatureEngineer":
        # and create additional features
        for col in columns:
            new_col = f"{col}_bin"
            labels = [f"Q_{i}" for i in range(bins)]
            self.df[new_col] = pd.cut(self.df[col], bins=bins, labels=labels)
        return self


    def ordinal_encode(self, columns: list) -> "FeatureEngineer":

        for col in columns:
            encoder=OrdinalEncoder()
            self.df[col]=encoder.fit_transform(self.df[[col]])
            self._encoders[col] = encoder
        return self



    def get_df(self) -> pd.DataFrame:
        return self.df



    def summary(self) -> None:
        print(f"Shape     : {self.df.shape}")
        print(f"Scaled    : {list(self._scalers.keys()) or 'none'}")
        print(f"Encoded   : {list(self._encoders.keys()) or 'none'}")
        print(f"Dtypes    :\n{self.df.dtypes}")


class Estimator:

    def __init__(self, model, test_size: float = 0.2, random_state: int = 42):
        self.model           = model
        self.test_size      = test_size
        self.random_state   = random_state

    # populated after each step
        self.X_train = self.X_test = None
        self.y_train = self.y_train = None
        self.y_pred = None
        self.metrics = {}

    def split(self, X: pd.DataFrame, y: pd.Series) -> "Estimator":
        '''Split data into train and test sets.'''
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(X, y, stratify = y, test_size=self.test_size, random_state=self.random_state)
    
    

In [82]:
WORK_DIR="/home/magnolia/DataScience/Stellar_Class"
TRAIN_PATH=Path(WORK_DIR,"data/train.csv")
TEST_PATH=Path(WORK_DIR, "data/test.csv")
MODEL_DIR=Path(WORK_DIR,"models")

df=pd.read_csv(TRAIN_PATH).set_index('id')
target=df['class']
X=df.drop(['class'], axis=1)
# map targets to numericals 

target_map={"GALAXY":0, "QSO": 1, "STAR": 2}
rev_map={0:"GALAXY", 1:"QSO", 2:"STAR"}

target=target.map(target_map)

In [ ]:
fe=FeatureEngineer(X)
NUM_COLS=fe.get_df().select_dtypes('float64').columns.to_list() # can define self.num and self.cat inside FeatureEngineer class 
CAT_COLS=fe.get_df().select_dtypes('object').columns.to_list()

(fe.
 minmax_scale(columns=NUM_COLS).
 onehot_encode(columns=CAT_COLS, drop_first=False).
 equal_width_bin(columns=NUM_COLS, bins=30))

BIN_COLS=fe.get_df().filter(regex=r'_bin$').columns.to_list()

fe.ordinal_encode(columns=BIN_COLS)

X=fe.get_df()

from sklearn.preprocessing import PolynomialFeatures
pf=PolynomialFeatures()
X=pf.fit_transform(X)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, target, stratify=target, random_state=42)

In [8]:
from sklearn.ensemble import HistGradientBoostingClassifier

model=HistGradientBoostingClassifier()
model.fit(X_train, y_train)

HistGradientBoostingClassifier()

In [36]:
y_pred=model.predict(X_train)

In [69]:
from sklearn.metrics import balanced_accuracy_score
balanced_accuracy_score(y_train, y_pred)

0.9517906916582451

In [108]:
# Prediction
df_pred=pd.read_csv(TEST_PATH).set_index('id')
df_pred.columns

Index(['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type',
       'galaxy_population'],
      dtype='object')

In [109]:
fe_pred=FeatureEngineer(df_pred)

(fe_pred.
 minmax_scale(columns=NUM_COLS).
 onehot_encode(columns=CAT_COLS, drop_first=False).
 equal_width_bin(columns=NUM_COLS, bins=30))

BIN_COLS=fe_pred.get_df().filter(regex=r'_bin$').columns.to_list()

fe_pred.ordinal_encode(columns=BIN_COLS)

X_pred=fe_pred.get_df()

from sklearn.preprocessing import PolynomialFeatures
pf=PolynomialFeatures()
X_pred=pf.fit_transform(X_pred)
y_pred=model.predict(X_pred)

In [110]:
y_pred=[rev_map[p] for p in y_pred]

In [118]:
pd.DataFrame({'id':df_pred.index, 'class':y_pred}).set_index('id').to_csv(Path(WORK_DIR, 'submission', 'submission2.csv'))

In [ ]:
# HORRIBLE PERFORMANCE BECAUSE MY TRAINING SET IS LEAKING INTO MY TEST SET OMG!!!!!!!!!!!!!!!!!!!!!!!
# I was applying the scale transformations all wrong. ;;;-;;;